# 01 — Cross-Dataset Feature Compatibility

**Goal:** Identify possible shared network features before cross-dataset testing.  
**Datasets:** X-IIoTID, ToN-IoT, Edge-IIoTset.  
**Rule:** Read only saved data-quality reports and dataset headers. **No training, no new split, no test scoring.**

A *candidate match* means names look similar. It does **not** prove two features measure the same thing.

## 1. Read existing dataset reports

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
from IPython.display import display

ROOT = Path(r'D:\J26-CS-336-Network-Flow-V2\network_flow_v2')
AUDITS = ROOT / 'reports' / 'audits'
METRICS = ROOT / 'reports' / 'metrics'
AUDITS.mkdir(parents=True, exist_ok=True)
METRICS.mkdir(parents=True, exist_ok=True)

sources = {
    'x_iiotid': {
        'audit': AUDITS / 'x_iiotid_v3_feature_quality.json',
        'processed': ROOT / 'data/processed/x_iiotid/x_iiotid_core19_selected_v3.csv.gz',
        'rows': 820834,
        'target': 'class3',
        'level': 'connection / flow candidate',
    },
    'ton_iot': {
        'audit': AUDITS / 'ton_iot_v3_feature_quality.json',
        'processed': ROOT / 'data/processed/ton_iot/ton_iot_core10_selected_v3.csv.gz',
        'rows': 211043,
        'target': 'label',
        'level': 'connection / flow candidate',
    },
    'edge_iiotset': {
        'audit': AUDITS / 'edge_iiotset_v3_feature_quality.json',
        'processed': ROOT / 'data/processed/edge_iiotset/edge_iiotset_core16_selected_v3.csv.gz',
        'rows': 157800,
        'target': 'Attack_label',
        'level': 'packet / protocol candidate',
    },
}

reports = {}
summary = []
for name, spec in sources.items():
    for key in ('audit', 'processed'):
        if not spec[key].is_file():
            raise FileNotFoundError(f'{name}: missing {key}: {spec[key]}')
    report = json.loads(spec['audit'].read_text(encoding='utf-8'))
    if int(report['rows']) != spec['rows']:
        raise ValueError(f'{name}: audit row count changed, investigate first.')
    selected = list(report['selected_features'])
    actual_header = pd.read_csv(spec['processed'], nrows=0).columns.tolist()
    absent = [f for f in selected if f not in actual_header]
    if absent or spec['target'] not in actual_header:
        raise ValueError(f'{name}: missing columns: {absent} / target {spec["target"]}')
    reports[name] = report
    summary.append({'dataset':name, 'rows':spec['rows'], 'selected_features':len(selected),
                    'target':spec['target'], 'record_type':spec['level'], 'header_valid':True})

print('Dataset metadata and selected-file headers verified. No data rows read.')
display(pd.DataFrame(summary))

## 2. Compare possible shared features

In [ ]:
# Only candidate comparisons. Do NOT automatically rename or combine datasets.
# ToN-IoT and X-IIoTID may differ in definitions, units and missing-value conventions.
proposals = [
    ('protocol', 'Protocol', 'proto', 'categorical'),
    ('service', 'Service', 'service', 'categorical'),
    ('flow_duration', 'Duration', 'duration', 'numeric'),
    ('source_payload_bytes', 'Scr_bytes', 'src_bytes', 'numeric'),
    ('destination_payload_bytes', 'Des_bytes', 'dst_bytes', 'numeric'),
    ('connection_state', 'Conn_state', 'conn_state', 'categorical'),
    ('source_packets', 'Scr_pkts', 'src_pkts', 'numeric'),
    ('source_ip_bytes', 'Scr_ip_bytes', 'src_ip_bytes', 'numeric'),
    ('destination_packets', 'Des_pkts', 'dst_pkts', 'numeric'),
    ('destination_ip_bytes', 'Des_ip_bytes', 'dst_ip_bytes', 'numeric'),
]

mapping = pd.DataFrame([
    {
        'concept':concept,
        'x_iiotid_column':x_col,
        'ton_iot_column':ton_col,
        'edge_iiotset_equivalent': 'NOT_VERIFIED',
        'kind':kind,
        'columns_exist':(x_col in reports['x_iiotid']['selected_features'] and
                         ton_col in reports['ton_iot']['selected_features']),
        'same_meaning_verified':False,
        'same_units_verified':False,
        'gateway_extractable_verified':False,
        'status':'CANDIDATE_ONLY',
    }
    for concept, x_col, ton_col, kind in proposals
])
if not mapping['columns_exist'].all():
    raise ValueError('Some proposed columns are absent from the selected features. Stop.')
display(mapping[['concept','x_iiotid_column','ton_iot_column','kind','status']])
print('10 candidate X-IIoTID ↔ ToN-IoT mappings; 0 verified semantic matches.')
print('Edge-IIoTset Core16 is packet/protocol oriented; direct flow equivalence NOT established.')

## 3. Review the risks before model transfer

- **Feature meaning:** Confirm how each dataset creates `duration`, packet counts and byte counts; similar names alone are insufficient.
- **Labels:** X-IIoTID uses `class3`; ToN-IoT uses `label`; Edge-IIoTset uses `Attack_label`. Never include label fields as inputs.
- **Missing values:** X-IIoTID has missing traffic measurements; ToN-IoT often has missing `service` values.
- **Shortcut risk:** Edge-IIoTset has strong TCP-port dependence and no MITM records in the current validation partition.
- **Split limitations:** Distinct feature patterns do not prove independent capture sessions or attacks.

**Decision:** Do **not** run cross-dataset transfer until shared feature definitions, units and a valid input transformation have been reviewed. We may later compare X-IIoTID → ToN-IoT using a separately trained *shared-feature* source model. The existing Core19 model cannot simply accept ToN-IoT Core10 columns.

## 4. Save two small reports

In [ ]:
csv_path = METRICS / 'cross_dataset_v3_feature_compatibility.csv'
json_path = AUDITS / 'cross_dataset_v3_feature_compatibility.json'
mapping.to_csv(csv_path, index=False)

result = {
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'purpose':'Pre-transfer feature compatibility audit, not a transfer score',
    'dataset_metadata':summary,
    'mapping_count':len(mapping),
    'semantically_verified_count':int(mapping['same_meaning_verified'].sum()),
    'approval':'NOT_APPROVED_FOR_TRANSFER',
    'candidate_mappings':mapping.to_dict(orient='records'),
    'warnings':[
        'Column-name similarity is not a semantic equivalence check.',
        'No held-out test files were read or scored.',
        'Edge-IIoTset packet/protocol records are not directly equivalent to connection flows.',
        'Capture-session independence has not been established.',
        'The current X-IIoTID Core19 model cannot directly accept ToN-IoT Core10 inputs.',
    ],
}
json_path.write_text(json.dumps(result, indent=2), encoding='utf-8')
print('Saved:',csv_path)
print('Saved:',json_path)
print('Next: verify shared feature semantics and units before creating a genuine transfer experiment.')